# NeuroGolf 2026 — Tiny ONNX Networks for ARC-AGI Puzzles

> **Kaggle Competition**: [NeuroGolf 2026 Championship](https://www.kaggle.com/competitions/neurogolf-2026)  
> **Repository**: [zikuanqi/NeuroGolf](https://github.com/zikuanqi/NeuroGolf)  
> **Current Score**: 1368.84 (98 / 400 tasks solved)

This notebook provides a complete walkthrough of the NeuroGolf solution framework — from setting up the environment, understanding the data representation, building custom pattern-specific solvers, to packaging and submitting to Kaggle.

## 1. Competition Overview

The **2026 NeuroGolf Championship** challenges participants to create the smallest possible neural networks that solve ARC-AGI (Abstraction and Reasoning Corpus) image transformation tasks.

### Key Rules

- **One ONNX network per task**: Each of the 400 ARC tasks requires its own `.onnx` model
- **Input/Output**: `(1, 10, 30, 30)` float32 one-hot encoded tensors (`"input"` / `"output"`)
- **All examples must pass**: If any single train/test/arc-gen example is wrong → 0 points
- **File size**: ≤ 1.44 MB per `.onnx` file
- **Banned ops**: `LOOP`, `SCAN`, `NONZERO`, `UNIQUE`, `SCRIPT`, `FUNCTION`, `COMPRESS`, `Sequence*`
- **Shapes**: All tensors must be statically inferable

### Scoring Formula

$$\text{points} = \max(1, 25 - \ln(\text{memory_bytes} + \text{params}))$$

Smaller networks = higher scores. A near-zero-memory `Transpose` (0 params) scores the full **25 points**.

## 2. Setup & Installation

In [ ]:
# Install dependencies
!pip install numpy>=1.26 onnx>=1.15 onnxruntime>=1.17 onnx_tool>=0.9 kaggle>=1.6

In [ ]:
import json
import math
import pathlib
from typing import Optional

import numpy as np
import onnx
import onnxruntime
from onnx import TensorProto, helper, numpy_helper

In [ ]:
# Download competition data (requires Kaggle API credentials)
# !kaggle competitions download -c neurogolf-2026 -p data/
# !unzip data/neurogolf-2026.zip -d data/

## 3. Data Representation: One-Hot Encoding

Each ARC grid is converted to a `(1, 10, 30, 30)` float32 tensor:

- **Channel `k`** = 1.0 where the cell has color `k` (0-9)
- The real grid sits in the top-left corner
- Cells outside the grid are all-zero padding
- Output is decoded by thresholding at `> 0.0`

In [ ]:
BATCH, CHANNELS, HEIGHT, WIDTH = 1, 10, 30, 30
TENSOR_SHAPE = (BATCH, CHANNELS, HEIGHT, WIDTH)


def to_onehot(grid: list[list[int]]) -> np.ndarray | None:
    """Convert a 2D ARC grid to (1,10,30,30) one-hot tensor."""
    if max(len(grid), len(grid[0])) > 30:
        return None
    out = np.zeros(TENSOR_SHAPE, dtype=np.float32)
    for r, row in enumerate(grid):
        for c, color in enumerate(row):
            out[0, color, r, c] = 1.0
    return out


def from_onehot(tensor: np.ndarray) -> list[list[int]]:
    """Convert (1,10,30,30) one-hot tensor back to 2D grid."""
    _, channels, height, width = tensor.shape
    rows: list[list[int]] = []
    for r in range(height):
        row: list[int] = []
        for c in range(width):
            colors = [k for k in range(channels) if tensor[0, k, r, c] == 1]
            if len(colors) == 1:
                row.append(colors[0])
            elif colors:
                row.append(11)  # ambiguous
            else:
                row.append(10)  # padding sentinel
        while row and row[-1] == 10:
            row.pop()
        rows.append(row)
    while rows and not rows[-1]:
        rows.pop()
    return rows


# Quick demo:
demo_grid = [[1, 2, 0], [0, 0, 3], [1, 0, 0]]
tensor = to_onehot(demo_grid)
print(f"Grid shape: {demo_grid}")
print(f"Tensor shape: {tensor.shape}")
print(f"Round-trip matches: {from_onehot(tensor) == demo_grid}")

## 4. How Solvers Work

The core idea: instead of training models, we build a **library of 77 hand-crafted, pattern-specific solvers**. Each solver:

1. **Detects** whether a task matches its pattern (returns `None` otherwise)
2. **Derives parameters** from examples (wall colors, scale factors, etc.)
3. **Builds** an exact ONNX graph with baked-in constants

Below is a minimal solver example for the **identity** pattern.

In [ ]:
from neurogolf.grids import CHANNELS, HEIGHT, WIDTH, all_examples

# --- ONNX graph builders ---

IR_VERSION = 10
OPSET_IMPORTS = [helper.make_opsetid("", 10)]
DATA_TYPE = TensorProto.FLOAT
GRID_SHAPE = [1, CHANNELS, HEIGHT, WIDTH]


def make_io():
    """Create standard input/output value_info."""
    x = helper.make_tensor_value_info("input", DATA_TYPE, GRID_SHAPE)
    y = helper.make_tensor_value_info("output", DATA_TYPE, GRID_SHAPE)
    return x, y


def finalize(nodes, initializers, name="graph"):
    """Assemble an ONNX model from nodes and initializers."""
    x, y = make_io()
    graph = helper.make_graph(nodes, name, [x], [y], initializers)
    return helper.make_model(
        graph, ir_version=IR_VERSION, opset_imports=OPSET_IMPORTS
    )


def identity_model() -> onnx.ModelProto:
    """Output equals input — 0 parameters, 0 memory."""
    node = helper.make_node("Identity", ["input"], ["output"])
    return finalize([node], [])

In [ ]:
def solve_identity(task: dict) -> Optional[onnx.ModelProto]:
    """Solver: does this task require output ≡ input?"""
    for ex in all_examples(task):
        if ex["input"] != ex["output"]:
            return None  # Pattern doesn't match
    return identity_model()

### 4.1 A More Complex Example: Gravity Down

This solver detects tasks where cells in each column "fall" straight down to the bottom. For each color in each column, the output fills the bottom `count(color, column)` rows.

In [ ]:
def _detect_gravity_down(task: dict) -> bool:
    """Verify: every column's cells sink to the bottom edge."""
    examples = list(all_examples(task))
    if not examples:
        return False
    for ex in examples:
        inp, out = ex["input"], ex["output"]
        h, w = len(inp), len(inp[0])
        if len(out) != h or len(out[0]) != w:
            return False
        for c in range(w):
            vals = [inp[r][c] for r in range(h) if inp[r][c] != 0]
            col = [0] * (h - len(vals)) + vals
            for r in range(h):
                if out[r][c] != col[r]:
                    return False
    return True


def solve_gravity_down_example(task: dict) -> Optional[onnx.ModelProto]:
    """Build ONNX graph for gravity-down transformation."""
    if not _detect_gravity_down(task):
        return None
    
    def i64(name, vals):
        return numpy_helper.from_array(np.array(vals, dtype=np.int64), name)
    def f32(name, arr):
        return numpy_helper.from_array(arr.astype(np.float32), name)
    
    rows = np.arange(HEIGHT, dtype=np.float32).reshape(1, 1, HEIGHT, 1)
    cols = np.arange(WIDTH, dtype=np.float32).reshape(1, 1, 1, WIDTH)
    
    init = [
        f32("zero_f", np.array([0.0])),
        f32("one_f", np.array([1.0])),
        f32("rows", rows),
        f32("cols", cols),
        i64("c1", [1]),
        i64("c10", [CHANNELS]),
        i64("axis_ch", [1]),
        i64("step1", [1]),
    ]
    
    nodes = [
        # Content mask & real grid dimensions
        helper.make_node("ReduceSum", ["input"], ["content"], axes=[1], keepdims=1),
        helper.make_node("ReduceMax", ["content"], ["row_has"], axes=[3], keepdims=1),
        helper.make_node("ReduceSum", ["row_has"], ["H_real"], axes=[2], keepdims=1),
        helper.make_node("ReduceMax", ["content"], ["col_has"], axes=[2], keepdims=1),
        helper.make_node("ReduceSum", ["col_has"], ["W_real"], axes=[3], keepdims=1),
        
        # Per-channel, per-column color counts
        helper.make_node("Greater", ["input", "zero_f"], ["mask_b"]),
        helper.make_node("Cast", ["mask_b"], ["mask"], to=TensorProto.FLOAT),
        helper.make_node("ReduceSum", ["mask"], ["count"], axes=[2], keepdims=1),
        
        # Fill rows [H_real - count, H_real)
        helper.make_node("Sub", ["H_real", "count"], ["low"]),
        helper.make_node("Less", ["rows", "low"], ["below_low_b"]),
        helper.make_node("Cast", ["below_low_b"], ["below_low"], to=TensorProto.FLOAT),
        helper.make_node("Sub", ["one_f", "below_low"], ["ge_low"]),
        helper.make_node("Less", ["rows", "H_real"], ["lt_h_b"]),
        helper.make_node("Cast", ["lt_h_b"], ["lt_h"], to=TensorProto.FLOAT),
        helper.make_node("Mul", ["ge_low", "lt_h"], ["sunk_all"]),
        
        # Channels 1-9 with channel-0 reconstructed as complement
        helper.make_node("Slice", ["sunk_all", "c1", "c10", "axis_ch", "step1"], ["colors"]),
        helper.make_node("ReduceSum", ["colors"], ["color_sum"], axes=[1], keepdims=1),
        helper.make_node("Less", ["rows", "H_real"], ["row_in_b"]),
        helper.make_node("Cast", ["row_in_b"], ["row_in"], to=TensorProto.FLOAT),
        helper.make_node("Less", ["cols", "W_real"], ["col_in_b"]),
        helper.make_node("Cast", ["col_in_b"], ["col_in"], to=TensorProto.FLOAT),
        helper.make_node("Mul", ["row_in", "col_in"], ["in_grid"]),
        helper.make_node("Sub", ["one_f", "color_sum"], ["not_color"]),
        helper.make_node("Mul", ["in_grid", "not_color"], ["bg"]),
        helper.make_node("Concat", ["bg", "colors"], ["output"], axis=1),
    ]
    
    x = helper.make_tensor_value_info("input", DATA_TYPE, GRID_SHAPE)
    y = helper.make_tensor_value_info("output", DATA_TYPE, GRID_SHAPE)
    graph = helper.make_graph(nodes, "gravity_down", [x], [y], initializer=init)
    return helper.make_model(graph, opset_imports=[helper.make_operatorsetid("", 11)], ir_version=8)

### 4.2 Solver Families Summary

The framework contains **77 solvers** organized into **9 families**:

| Family | Examples | Key ops |
|---|---|---|
| **1. Rigid Moves** | identity, transpose, shift, shape-aware flip/rotate | `Identity`, `Transpose`, `Gather`, `Slice` |
| **2. Crop & Extract** | static crop, marker crop, bbox strip | `Slice`, `Pad`, `ArgMax`, `ReduceSum` |
| **3. Scale & Tile** | Kron scale, resize, tiling, palindrome mirror | `Gather`, `Resize`, `Tile`, `Pad` |
| **4. Recolor** | remap, single-color, majority fill, blob recolor | `Conv`, `ReduceSum`, `Where`, `OneHot` |
| **5. Lines & Gravity** | connect dots, gravity down/up/right, flood fill | `CumSum`, `Conv`, `Slice`, `Concat` |
| **6. Morphology** | dilate, outline, denoise, stamp, mirror complete | `Conv`, `Greater`, `Mul`, `Concat` |
| **7. Connected Components** | isolate recolor, cc size/rank recolor | `Pad`+`Conv` unrolled propagation |
| **8. Counting & Logic** | count bar, split AND/logic, learned conv | `Conv`, `ReduceSum`, `Sub`, `Mul` |
| **9. Classification** | symmetry/shape/count classification | feature hash → hardcoded lookup table |

## 5. Verification: The Clean-Room Scorer

The scorer (`verify.py`) loads the ONNX model, runs it through ONNX Runtime on **all** examples, and checks for exact match. It also enforces all competition constraints.

In [ ]:
def verify_model(model_path: str, task: dict) -> dict:
    """Simplified verification: run model on all examples."""
    path = pathlib.Path(model_path)
    if not path.is_file():
        return {"passed": False, "error": "file not found"}
    
    # Check file size
    if path.stat().st_size > int(1.44 * 1024 * 1024):
        return {"passed": False, "error": "file too large"}
    
    model = onnx.load(str(path))
    session = onnxruntime.InferenceSession(model.SerializeToString())
    
    results = {"train": {"right": 0, "wrong": 0}, "test": {"right": 0, "wrong": 0}}
    
    for split in ["train", "test"]:
        for ex in task.get(split, []):
            inp = to_onehot(ex["input"])
            expected = to_onehot(ex["output"])
            if inp is None or expected is None:
                continue
            result = session.run(["output"], {"input": inp})
            predicted = (result[0] > 0.0).astype(np.float32)
            if np.array_equal(predicted, expected):
                results[split]["right"] += 1
            else:
                results[split]["wrong"] += 1
    
    passed = results["train"]["wrong"] == 0 and results["test"]["wrong"] == 0
    return {"passed": passed, "results": results}


# Compute score using the official formula
def compute_score(memory_bytes: int, params: int) -> float:
    return max(1.0, 25.0 - math.log(max(1.0, memory_bytes + params)))

## 6. The Pipeline: Building All Tasks

`pipeline.build_one(task_num, task, out_dir)`:

1. Iterates through all 77 solvers
2. Each solver either returns an ONNX model or `None`
3. The model is verified with the clean-room scorer
4. The **highest-scoring** passing model is saved as `taskNNN.onnx`

In [ ]:
# Run the full pipeline (this takes several minutes)
# !python scripts/build_all.py

# Build a single task for testing:
# !python scripts/build_all.py --from 32 --to 32

# Batch build a range:
# !python scripts/build_all.py --from 1 --to 100

## 7. Results Analysis

After running `build_all.py`, the results are stored in `networks/build_summary.json`.

In [ ]:
# Analyze build results
import json
from collections import Counter

summary = json.load(open("networks/build_summary.json"))

solved = [e for e in summary if e.get("solver") != "none" and e.get("points", 0) > 0]
unsolved = [e for e in summary if e.get("solver") == "none" or e.get("points", 0) == 0]

print(f"{'='*60}")
print(f"  Solved: {len(solved):3d} / {len(summary)} tasks")
print(f"  Total score: {sum(e['points'] for e in solved):.2f}")
print(f"{'='*60}")

# Solver distribution
solver_counts = Counter(e["solver"] for e in solved)
print(f"\n{'Solver':35s} {'Tasks':>5s} {'Total Pts':>10s}")
print("-" * 52)
for name, count in solver_counts.most_common():
    total_pts = sum(e["points"] for e in solved if e["solver"] == name)
    print(f"{name:35s} {count:5d} {total_pts:10.2f}")

In [ ]:
# Top scoring tasks (simplest = highest score)
print(f"\n{'Task':>6s}  {'Solver':30s}  {'Points':>8s}  {'Memory':>10s}  {'Params':>8s}")
print("-" * 74)
for e in sorted(solved, key=lambda x: x["points"], reverse=True)[:15]:
    print(f"{e['task']:6d}  {e['solver']:30s}  {e['points']:8.2f}  {e['memory']:10d}  {e['params']:8d}")

## 8. Packaging & Submitting to Kaggle

The submission must be a `submission.zip` containing all `taskNNN.onnx` files.

In [ ]:
# Package all .onnx files into submission.zip
import zipfile
import pathlib

networks = sorted(pathlib.Path("networks").glob("task*.onnx"))
submissions_dir = pathlib.Path("submissions")
submissions_dir.mkdir(exist_ok=True)

out_path = submissions_dir / "submission.zip"
with zipfile.ZipFile(out_path, "w", zipfile.ZIP_DEFLATED) as z:
    for n in networks:
        z.write(n, arcname=n.name)

print(f"Packaged {len(networks)} networks → {out_path} ({out_path.stat().st_size:,} bytes)")

In [ ]:
# Submit to Kaggle
# !kaggle competitions submit -c neurogolf-2026 -f submissions/submission.zip -m "My submission"

## 9. Adding a New Solver

The standard recipe for adding a new pattern solver:

```
1. Create src/neurogolf/solvers/my_rule.py
   ├── _detect(task) → bool        # Confirm pattern matches ALL examples
   ├── _build(params) → ModelProto # Assemble ONNX graph
   └── solve_my_rule(task) → Optional[ModelProto]
2. Register in solvers/__init__.py
3. Add positive + negative test cases
4. Build & verify: python scripts/build_all.py --from N --to N
```

### Key Constraints for New Solvers

- **No loops**: Use fixed-round unrolled iterations (`Pad` + `Conv` + `Max` rounds)
- **Static shapes**: Every intermediate tensor must be statically shaped
- **Clean one-hot output**: Exactly one channel `> 0` per real cell
- **Opset 10+**: Framework uses opset 10 as baseline; some solvers use opset 11

## 10. Roadmap & Future Work

| Status | Area | Description |
|---|---|---|
| ✅ Done | Small-grid classification | Family 9 solves tasks 56/103/167/186/262 using feature hashing |
| ✅ Done | Connected components | CC labeling, ranking, and counting via unrolled propagation |
| ✅ Done | Flood fill | Border-BFS with fixed 58-round unroll |
| 🔲 Open | Remaining 1×1 tasks | 48/291/346/355 — need clean statically-expressible hash |
| 🔲 Open | Object matching/copying | Locate a shape and stamp it elsewhere |
| 🔲 Open | Multi-step composition | Chain several primitive operations |
| 🔲 Open | Memory trimming | Fuse (1,10,30,30) intermediates in CC solvers |

## 11. Key Learnings

1. **Pattern-first approach works**: 98/400 tasks solved with hand-crafted, zero-training solvers
2. **ONNX as a programming language**: The banned ops (`LOOP`, `SCAN`) mean you must express all iteration as fixed unrolled computation
3. **Exactness is essential**: One wrong example = 0 points. Detection must be conservative.
4. **Memory matters**: The `ln(memory + params)` scoring means 1MB vs 1KB is ~6.9 points difference
5. **Opset 10-11 is expressive enough**: `Conv`, `Gather`, `Slice`, `ReduceSum`, `ArgMax`, `CumSum`, `TopK`, `And`/`Or` cover most ARC transformations